# 5.13 Dynamic Bayesian networks

A DBN repeats a local two-slice graph, sharing structure across time.

DBNs generalize Bayesian networks to temporal processes and contain HMMs and Kalman filters as special cases. Here we build exact unrolled filtering, then compare it with approximate or insufficient-state variants on a D1-D5 ladder.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 513
rng = np.random.default_rng(SEED)


def normalize(v):
    v = np.asarray(v, dtype=float)
    total = v.sum()
    if total <= 0:
        raise ValueError("cannot normalize nonpositive mass")
    return v / total


def dbn_filter(initial, transition, evidence_likelihoods):
    marginals = [normalize(initial)]
    current = normalize(initial)
    for likelihood in evidence_likelihoods:
        predicted = current @ transition
        current = normalize(predicted * likelihood)
        marginals.append(current)
    return np.vstack(marginals)


def dbn_predict(initial, transition, steps):
    current = normalize(initial)
    out = [current]
    for _ in range(steps):
        current = current @ transition
        out.append(current)
    return np.vstack(out)


def exact_unrolled(initial, transitions, evidence_likelihoods):
    current = normalize(initial)
    out = [current]
    for t, likelihood in enumerate(evidence_likelihoods):
        predicted = current @ transitions[t]
        current = normalize(predicted * likelihood)
        out.append(current)
    return np.vstack(out)


def marginal_error(estimated, exact):
    return float(np.mean(np.abs(estimated - exact)))


def build_dbn_ladder():
    d1_transition = np.array([[0.8, 0.2], [0.25, 0.75]])
    d1_evidence = [np.array([0.9, 0.4]) for _ in range(5)]
    d1 = {
        "name": "D1 two-state exact DBN",
        "initial": np.array([0.7, 0.3]),
        "transition": d1_transition,
        "evidence": d1_evidence,
        "exact_transitions": [d1_transition for _ in d1_evidence],
    }
    d2_transition = np.array([
        [0.70, 0.20, 0.08, 0.02],
        [0.15, 0.65, 0.15, 0.05],
        [0.05, 0.20, 0.60, 0.15],
        [0.02, 0.08, 0.20, 0.70],
    ])
    d2_evidence = [np.array([0.8, 0.7, 0.3, 0.2]), np.array([0.2, 0.4, 0.8, 0.7]), np.array([0.7, 0.8, 0.4, 0.2]), np.array([0.2, 0.3, 0.7, 0.9]), np.array([0.8, 0.5, 0.4, 0.2])]
    d2 = {
        "name": "D2 four-state tied template",
        "initial": normalize([0.55, 0.20, 0.15, 0.10]),
        "transition": d2_transition,
        "evidence": d2_evidence,
        "exact_transitions": [d2_transition for _ in d2_evidence],
    }
    d3_transition = np.array([[0.72, 0.20, 0.08], [0.18, 0.64, 0.18], [0.08, 0.20, 0.72]])
    d3_evidence = [np.array([0.9, 0.1, 0.8]), np.array([0.8, 0.2, 0.9]), np.array([0.2, 0.9, 0.2]), np.array([0.85, 0.15, 0.80]), np.array([0.15, 0.85, 0.25]), np.array([0.80, 0.20, 0.90])]
    d3 = {
        "name": "D3 ambiguous bimodal evidence",
        "initial": normalize([0.45, 0.10, 0.45]),
        "transition": d3_transition,
        "evidence": d3_evidence,
        "exact_transitions": [d3_transition for _ in d3_evidence],
    }
    d4_transition = np.array([[0.78, 0.18, 0.04], [0.22, 0.58, 0.20], [0.05, 0.20, 0.75]])
    d4_evidence = [np.array([30, 18, 5]), np.array([14, 26, 12]), np.array([7, 21, 28]), np.array([26, 16, 9]), np.array([10, 24, 23]), np.array([6, 18, 31])]
    d4 = {
        "name": "D4 traffic contingency table",
        "initial": normalize([60, 30, 10]),
        "transition": d4_transition,
        "evidence": [normalize(row) for row in d4_evidence],
        "exact_transitions": [d4_transition for _ in d4_evidence],
    }
    d5_transition = np.array([
        [0.65, 0.20, 0.05, 0.05, 0.03, 0.02],
        [0.15, 0.55, 0.15, 0.05, 0.05, 0.05],
        [0.05, 0.15, 0.55, 0.15, 0.05, 0.05],
        [0.04, 0.06, 0.18, 0.52, 0.15, 0.05],
        [0.03, 0.05, 0.06, 0.16, 0.55, 0.15],
        [0.02, 0.04, 0.06, 0.08, 0.20, 0.60],
    ])
    d5_evidence = [normalize(0.2 + rng.random(6)) for _ in range(12)]
    d5_transitions = []
    for t in range(len(d5_evidence)):
        drift = np.eye(6) * (0.01 if t % 3 == 0 else 0.0)
        d5_transitions.append(normalize_rows(d5_transition + drift))
    d5 = {
        "name": "D5 higher-dim DBN with sufficient-state issue",
        "initial": normalize([0.45, 0.20, 0.12, 0.10, 0.08, 0.05]),
        "transition": d5_transition,
        "evidence": d5_evidence,
        "exact_transitions": d5_transitions,
    }
    return [d1, d2, d3, d4, d5]


def normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=float)
    return matrix / matrix.sum(axis=1, keepdims=True)


def collapse_to_small_state(marginals):
    collapsed = np.column_stack([marginals[:, :3].sum(axis=1), marginals[:, 3:].sum(axis=1)])
    return collapsed


## The concept, built once: DBN filtering

The first-order DBN template repeats

$$p(x_{1:T})=p(x_1)\prod_{t=2}^{T}p(x_t\mid x_{t-1}).$$

The key operations are prediction through the transition matrix, evidence multiplication, and slice-wise normalization.

In [ ]:

initial = np.array([0.7, 0.3])
transition = np.array([[0.8, 0.2], [0.25, 0.75]])
prediction = initial @ transition
print("one-step prediction", prediction)
assert np.allclose(prediction, [0.635, 0.365])

tied_params = 2
untied_params = (10 - 1) * 2
print("tied vs untied", tied_params, untied_params)
assert tied_params == 2
assert untied_params == 18


Now include a sensor likelihood. Evidence is not another prior; it multiplies the predicted state and must be renormalized to become a probability vector.

In [ ]:

sensor_likelihood = np.array([0.9, 0.4])
posterior = normalize(prediction * sensor_likelihood)
manual = prediction * sensor_likelihood
manual = manual / manual.sum()
print("filtered posterior", posterior)
assert np.allclose(posterior, manual)
assert np.isclose(posterior.sum(), 1.0)


## The dataset ladder

We keep the same filtering method and increase state, evidence, and model mismatch complexity from D1 to D5.

In [ ]:

ladder = build_dbn_ladder()
for i, rung in enumerate(ladder, start=1):
    print(i, rung["name"])
    print("initial shape", rung["initial"].shape)
    print("transition shape", rung["transition"].shape)
    print("time steps", len(rung["evidence"]))
    print("first evidence", np.round(rung["evidence"][0], 3))


In [ ]:

rows = []
all_estimated = []
all_exact = []
for i, rung in enumerate(ladder, start=1):
    estimated = dbn_filter(rung["initial"], rung["transition"], rung["evidence"])
    exact = exact_unrolled(rung["initial"], rung["exact_transitions"], rung["evidence"])
    error = marginal_error(estimated, exact)
    rows.append((f"D{i}", rung["name"], error))
    all_estimated.append(estimated)
    all_exact.append(exact)

print("rung | marginal error")
for label, name, error in rows:
    print(f"{label} | {name} | {error:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, rung in enumerate(ladder):
    ax = axes[0, i]
    exact = all_exact[i]
    estimated = all_estimated[i]
    ax.plot(exact[:, 0], label="exact s0")
    ax.plot(estimated[:, 0], "--", label="estimated s0")
    ax.set_title(f"D{i + 1}")
    ax.set_xlabel("time")
    ax.set_ylabel("marginal")
    if i == 0:
        ax.legend(fontsize=8)

ax = axes[1, 0]
for i, exact in enumerate(all_exact):
    err_t = np.mean(np.abs(all_estimated[i] - exact), axis=1)
    ax.plot(err_t, marker="o", label=f"D{i + 1}")
ax.set_title("marginal-error-vs-time curve")
ax.set_xlabel("time")
ax.set_ylabel("mean absolute marginal error")
ax.legend(fontsize=8)
for j in range(1, 5):
    axes[1, j].axis("off")
plt.tight_layout()


## Pitfall on D5: state too small or untying the wrong thing

If the slice omits future-relevant variables, the Markov property fails. We reproduce that by collapsing six states to two groups before forecasting, then fix it with the sufficient six-state template and shared parameters.

In [ ]:

d5 = ladder[-1]
full = exact_unrolled(d5["initial"], d5["exact_transitions"], d5["evidence"])
collapsed_initial = np.array([d5["initial"][:3].sum(), d5["initial"][3:].sum()])
collapsed_transition = np.array([[0.75, 0.25], [0.20, 0.80]])
collapsed_evidence = []
for likelihood in d5["evidence"]:
    collapsed_evidence.append(np.array([likelihood[:3].mean(), likelihood[3:].mean()]))
small = dbn_filter(collapsed_initial, collapsed_transition, collapsed_evidence)
full_collapsed = collapse_to_small_state(full)
wrong_error = marginal_error(small, full_collapsed)
fixed_error = marginal_error(collapse_to_small_state(dbn_filter(d5["initial"], d5["transition"], d5["evidence"])), full_collapsed)
print("too-small-state error", wrong_error)
print("sufficient-state tied-template error", fixed_error)
assert fixed_error < wrong_error


## Evaluate it + practice

- Metric: mean absolute marginal error versus exact unrolled inference.
- No-skill baseline: carry the previous marginal forward without evidence updates.
- Cheap sanity check: every filtered slice sums to one and D1 prediction equals [0.635, 0.365].
- Ablation: remove evidence normalization or collapse D5 states and the error rises.
- Failure signals: posterior mass does not normalize, D5 group forecasts are biased, or filtering is confused with smoothing.

**Practice.** Change one D3 evidence vector and inspect how quickly the posterior recovers.

**Practice.** Add a backward smoothing pass and compare it with filtering on the same D2 sequence.

**Practice.** Tie and untie a D4 transition template and compare parameter counts.